## Stage 1.2 – Layout-Aware OCR Extraction from PO Images

This stage performs **high-fidelity OCR** on scanned Purchase Order images using a **Vision–Language Model (VLM)**.  
The objective is to extract **exactly visible text** (no inference or normalization) while preserving layout, order, and tabular structure.

---

### Input
- Page-wise PO images (`.jpg`, `.png`)
- Scanned from Purchase Order PDFs



In [1]:
import torch
import os
import json
import re
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image

MODEL_ID = "nanonets/Nanonets-OCR2-3B"
IMAGE_DIR = "/kaggle/input/hpcl-po-images"
OUTPUT_JSON = "/kaggle/working/ocr_output.json"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval()

print("✅ Model loaded")

def split_header_body(image, header_ratio=0.35):
    w, h = image.size
    header_h = int(h * header_ratio)
    return (
        image.crop((0, 0, w, header_h)),
        image.crop((0, header_h, w, h))
    )

def clean_ocr_text(text):
    lines = [l.rstrip() for l in text.splitlines()]
    lines = [l for l in lines if l.strip()]
    return "\n".join(lines)

def markdown_table_to_html(text):
    lines = text.splitlines()
    output = []
    i = 0

    while i < len(lines):
        line = lines[i]

        # detect markdown table
        if (
            line.strip().startswith("|")
            and i + 1 < len(lines)
            and "---" in lines[i + 1]
        ):
            headers = [h.strip() for h in line.strip("|").split("|")]
            i += 2 

            rows = []
            while i < len(lines) and lines[i].strip().startswith("|"):
                row = [c.strip() for c in lines[i].strip("|").split("|")]
                rows.append(row)
                i += 1

            # build HTML table
            html = ["<table>", "  <thead>", "    <tr>"]
            for h in headers:
                html.append(f"      <th>{h}</th>")
            html += ["    </tr>", "  </thead>", "  <tbody>"]

            for row in rows:
                html.append("    <tr>")
                for cell in row:
                    html.append(f"      <td>{cell}</td>")
                html.append("    </tr>")

            html += ["  </tbody>", "</table>"]
            output.extend(html)
        else:
            output.append(line)
            i += 1

    return "\n".join(output)

def run_ocr(img):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        "Extract ONLY the text exactly as visible in this image.\n"
                        "- Preserve line breaks and ordering.\n"
                        "- Do NOT infer, summarize, or normalize.\n"
                        "- Do NOT repeat text.\n"
                        "- If a table is visible, output it as plain text rows.\n\n"
                        "Return plain text only."
                    )
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    inputs = processor(
        images=img,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            repetition_penalty=1.15
        )

    text = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return clean_ocr_text(text)

image_files = sorted([
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

if not image_files:
    raise RuntimeError("❌ No images found")

final_json = {
    "document_type": "UNKNOWN_PO",
    "ocr_engine": MODEL_ID,
    "pages": []
}

for idx, image_path in enumerate(image_files, start=1):
    print(f"\n🔍 Processing: {image_path}")

    image = Image.open(image_path).convert("RGB")
    header_img, body_img = split_header_body(image)

    header_text = run_ocr(header_img)
    body_text = run_ocr(body_img)

    body_text = markdown_table_to_html(body_text)

    print("\n===== OCR OUTPUT =====\n")
    if header_text:
        print(header_text)
        print("\n")
    print(body_text)

    final_json["pages"].append({
        "page_number": idx,
        "image_file": os.path.basename(image_path),
        "header_text": header_text,
        "body_text": body_text,
        "full_text": (header_text + "\n\n" + body_text).strip()
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(final_json, f, indent=2, ensure_ascii=False)

print(f"\n✅ OCR JSON saved at: {OUTPUT_JSON}")



2026-01-17 20:32:33.121206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768681953.308016      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768681953.360614      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768681953.817216      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768681953.817254      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768681953.817257      23 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model loaded

🔍 Processing: /kaggle/input/hpcl-po-images/po1.jpeg

===== OCR OUTPUT =====

<img>HP logo</img>
हिन्दुस्तान पेट्रोलियम कॉरपोरेशन लिमिटेड
(एक सरकारी कंपनी)
Hindustan Petroleum Corporation Limited
A GOVERNMENT OF INDIA ENTERPRISE REGISTERED OFFICE: 17, JAMSHEDA TATA ROAD, MUMBAI -400 020.
593/1001, भवन विक्रयांग, एफ., एच. मोड़ेजा बाजाया, गाँव चोटी खास (ख.) , मुंबई - 400 013.
A: 803/101, MARATHAWALA FUTURE N. M. JOHD MAJOR GROUND, LOWER PAVILION, E. MANGAN - 400 013.
KOLHI / TEL : 2633 0000, g.h.india@petrolin.com CIN NO.: L33201MH1995PLC0008858
LETTER OF INTENT
Ref: PROC/E&I/CPO Date: 03.09.2019
To,
M/S RAMA CYLINDERS PVT. LTD. (24023B04)
SURVEY NO. 334, P1, 335,
VILL BHIMASAR, TAL ANJAR
KACHCHHL
PUNE-4102340


GJ01645 37824
e-mail: milind.khadke@ramacyinders.in, mrt@ramacyinders.in
Sub: SUPPLY OF CNG STORAGE CASCADE FOR CITY GAS DISTRIBUTION PROJECT IN JIND-SONIPAT GA
Ref: Your Offer against our Tender No. **19000045-HD-10157** Dt: **25.04.2019**
Sir/Madam,
We are please

##  Stage 1.3 – Raw OCR JSON → Structured PO JSON

This stage transforms noisy OCR output into a **clean, validated, and computation-ready structured JSON** representation of a Purchase Order.

It acts as the **final normalization layer** of the digitization pipeline before downstream item standardization and cost intelligence models.

---

### Input
- Page-wise OCR output in raw JSON format
- Each page contains unstructured text extracted from scanned PO PDFs

```json
{
  "pages": [
    {
      "full_text": "Raw OCR extracted text..."
    }
  ]
}


In [2]:
import json
import re

INPUT_FILE = "/kaggle/working/ocr_output.json"
OUTPUT_FILE = "/kaggle/working/struct_ocr.json"

def l1_clean(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"<img>.*?</img>", "", text, flags=re.DOTALL)
    text = re.sub(r"\[Signature.*?\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<signature>.*?</signature>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\*+", "", text)
    return "\n".join([l.strip() for l in text.splitlines() if l.strip()])

def parse_document(text):
    return {
        "document_type": "LETTER OF INTENT" if "LETTER OF INTENT" in text.upper() else None,
        "reference_number": re.search(r"Ref:\s*([A-Z0-9/&-]+)", text).group(1)
            if re.search(r"Ref:\s*([A-Z0-9/&-]+)", text) else None,
        "reference_date": re.search(r"Date:\s*([\d.]+)", text).group(1)
            if re.search(r"Date:\s*([\d.]+)", text) else None,
        "tender_number": re.search(r"Tender No\.\s*([A-Z0-9-]+)", text).group(1)
            if re.search(r"Tender No\.\s*([A-Z0-9-]+)", text) else None
    }


def parse_buyer(text):
    name = re.search(r"HINDUSTAN PETROLEUM CORPORATION LIMITED", text, re.IGNORECASE)
    office = re.search(r"REGISTERED OFFICE:\s*(.+)", text, re.IGNORECASE)

    return {
        "name": name.group(0) if name else None,
        "registered_office": office.group(1).strip() if office else None
    }


def parse_vendor(text):
    m = re.search(r"M/S\s+(.+?)\s*\(([\w\d]+)\)", text)
    return {
        "name": m.group(1).strip() if m else None,
        "vendor_code": m.group(2) if m else None
    }


def parse_commercial(text):
    return {
        "order_value_raw": re.search(r"value of Rs\.\s*([\d,]+)", text).group(1)
            if re.search(r"value of Rs\.\s*([\d,]+)", text) else None,
        "security_deposit_raw": re.search(r"Security Deposit for Rs\.\s*([\d,]+)", text).group(1)
            if re.search(r"Security Deposit for Rs\.\s*([\d,]+)", text) else None,
        "security_deposit_days": int(re.search(r"within\s*(\d+)\s*Days", text).group(1))
            if re.search(r"within\s*(\d+)\s*Days", text) else None,
        "bg_percentage": re.search(r"@\s*(\d+%)", text).group(1)
            if re.search(r"@\s*(\d+%)", text) else None
    }


def parse_signatory(text):
    m = re.search(r"\n([A-Z][a-z]+ [A-Z][a-z]+)\n(Ch\..+)", text)
    return {
        "name": m.group(1) if m else None,
        "designation": m.group(2) if m else None
    }


def parse_annexure(text):
    rows = re.findall(
        r"<tr>\s*<td>(.*?)</td>\s*<td>(\d+)</td>\s*<td>(.*?)</td>\s*<td>([\d,]+)</td>\s*<td>(\d+%)</td>",
        text,
        re.DOTALL
    )

    items = []
    for d, q, u, r, g in rows:
        items.append({
            "description": d.strip(),
            "quantity_raw": q,
            "uom": u.strip(),
            "unit_rate_raw": r,
            "gst_raw": g
        })

    return items

def normalize_money(val):
    return int(val.replace(",", "")) if val else None


def l3_normalize(data):
    data["commercial"]["order_value"] = normalize_money(
        data["commercial"]["order_value_raw"]
    )
    data["commercial"]["security_deposit"] = normalize_money(
        data["commercial"]["security_deposit_raw"]
    )

    for item in data["annexure"]:
        item["quantity"] = int(item["quantity_raw"])
        item["unit_rate"] = normalize_money(item["unit_rate_raw"])
        item["gst_percent"] = int(item["gst_raw"].replace("%", ""))

    return data

def l4_intelligence(data):
    total = 0
    for item in data["annexure"]:
        base = item["quantity"] * item["unit_rate"]
        gst = base * item["gst_percent"] / 100
        item["line_total"] = base + gst
        total += item["line_total"]

    data["financial_summary"] = {
        "computed_po_value": total,
        "high_value_po": total > 5_000_000,
        "security_deposit_present": data["commercial"]["security_deposit"] is not None
    }

    return data

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    ocr = json.load(f)

full_text = "\n".join(
    l1_clean(p["full_text"]) for p in ocr["pages"]
)

structured = {
    "document": parse_document(full_text),
    "buyer": parse_buyer(full_text),
    "vendor": parse_vendor(full_text),
    "commercial": parse_commercial(full_text),
    "signatory": parse_signatory(full_text),
    "annexure": parse_annexure(full_text)
}

structured = l3_normalize(structured)
structured = l4_intelligence(structured)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(structured, f, indent=2, ensure_ascii=False)

print(f"✅ Structured output written to {OUTPUT_FILE}")



✅ Structured output written to /kaggle/working/struct_ocr.json


In [3]:
import json
import pandas as pd

json_path = "/kaggle/input/jsobn-read/struct_ocr.json"

with open(json_path, "r") as f:
    data = json.load(f)

rows = []

for item in data["annexure"]:
    basic_amount = item["quantity"] * item["unit_rate"]
    gst_amount = basic_amount * item["gst_percent"] / 100

    row = {
        "po_id": data["document"]["tender_number"],
        "region": "NA",                         # excluded deriving regions from given city info
        "department": "NA",      #excluding these details for now
        "vendor": data["vendor"]["name"],
        "item_description": item["description"],
        "quantity": item["quantity"],
        "unit": item["uom"],
        "basic_rate": item["unit_rate"],
        "basic_amount": basic_amount,
        "gst_percent": item["gst_percent"],
        "gst_amount": gst_amount,
        "total_amount": item["line_total"]
    }

    rows.append(row)

df = pd.DataFrame(rows)

df


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/jsobn-read/struct_ocr.json'